In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:

# ============================================================
# IDMC TO DATABRICKS - MIGRATION + EXCEL REPORT
# HUMAN GATEWAY APPROVAL INTEGRATION
# ============================================================

import os
import json
import importlib

import pipeline.migration_pipeline as migration_pipeline_module

from report.excel_report_generator import generate_excel_report

# Reload the updated pipeline module
importlib.reload(migration_pipeline_module)

MigrationPipeline = migration_pipeline_module.MigrationPipeline


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"
)

METADATA_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/parsed_metadata/"
    "idmc_parsed_objects.json"
)

OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

REVIEW_DIR = os.path.join(OUTPUT_DIR, "human_reviews")

WORKSPACE_URL = (
    "https://dbc-ffb099c2-fea8.cloud.databricks.com"
)

LLM_MODEL = "databricks-gpt-oss-120b"


# ------------------------------------------------------------
# 2. EXPLICIT HUMAN GATEWAY APPROVALS
# ------------------------------------------------------------

APPROVED_REVIEW_IDS = {
    "m_customer_standardize":
        "af28cd97-0156-41aa-903c-a23f02660a1f",

    "m_order_enrichment":
        "e71c10f9-3d7a-4762-9b08-3eb1fc31496c",

    "m_inventory_reconciliation":
        "ac992826-72c1-4512-8dcc-fc11dafd615e",

    "m_daily_sales_summary":
        "e1557494-784a-4aa4-96f0-79abc89dc375"
}


# ------------------------------------------------------------
# 3. SAFE EXECUTION SETTINGS
# ------------------------------------------------------------

DRY_RUN = True
EXECUTION_APPROVED = False
ALLOW_SYNTHETIC_MERGE_KEYS = False

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isfile(XML_PATH):
    raise FileNotFoundError(XML_PATH)

if not os.path.isdir(REVIEW_DIR):
    raise FileNotFoundError(
        f"Human Gateway review directory not found: {REVIEW_DIR}"
    )


# ------------------------------------------------------------
# 4. DATABRICKS AUTHENTICATION
# ------------------------------------------------------------

DATABRICKS_TOKEN = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

if not DATABRICKS_TOKEN:
    raise ValueError("Databricks token unavailable.")


# ------------------------------------------------------------
# 5. DISPLAY APPROVAL CONFIGURATION
# ------------------------------------------------------------

print("=" * 90)
print("IDMC MIGRATION - APPROVAL INTEGRATION")
print("=" * 90)

print("Pipeline:", MigrationPipeline)
print("Selected mapping count:", len(APPROVED_REVIEW_IDS))

for mapping, review_id in APPROVED_REVIEW_IDS.items():
    print(mapping, "=>", review_id)


# ------------------------------------------------------------
# 6. INITIALIZE MIGRATION PIPELINE
# ------------------------------------------------------------

pipeline = MigrationPipeline(
    xml_path=XML_PATH,
    spark=spark,
    output_dir=OUTPUT_DIR,
    workspace_url=WORKSPACE_URL,
    databricks_token=DATABRICKS_TOKEN,
    llm_model=LLM_MODEL,

    selected_mapping_name=None,

    dry_run=DRY_RUN,
    execution_approved=EXECUTION_APPROVED,
    allow_synthetic_merge_keys=ALLOW_SYNTHETIC_MERGE_KEYS,

    verbose=True,
    human_review_dir=REVIEW_DIR,

    # NEW: mapping-specific Human Gateway approvals
    approved_review_ids=APPROVED_REVIEW_IDS
)


# ------------------------------------------------------------
# 7. RUN MIGRATION PIPELINE
# ------------------------------------------------------------

print("\nSTARTING MIGRATION PIPELINE")

migration_results = pipeline.run()

if not isinstance(migration_results, list):
    raise TypeError("MigrationPipeline must return a list.")

if not migration_results:
    raise RuntimeError("No migration results returned.")


# ------------------------------------------------------------
# 8. DISPLAY MIGRATION RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("MIGRATION RESULTS")
print("=" * 90)

for result in migration_results:

    print("\nMapping:", result.get("mapping_name"))
    print("Success:", result.get("success"))
    print("Stage:", result.get("stage"))
    print("Validation:", result.get("validation_status"))
    print("Executed:", result.get("executed"))

    print(
        "Human approval verified:",
        result.get("human_approval_verified")
    )

    print(
        "Approval review ID:",
        result.get("approval_review_id")
    )

    print("Error:", result.get("error"))


# ------------------------------------------------------------
# 9. GENERATE EXCEL REPORT
# ------------------------------------------------------------

print("\nGENERATING EXCEL REPORT")

report_paths = generate_excel_report(
    migration_results=migration_results,
    xml_path=XML_PATH,
    metadata_path=METADATA_PATH,
    output_dir=OUTPUT_DIR,
    spark=spark,
    dbutils=dbutils
)


# ------------------------------------------------------------
# 10. VERIFY REPORT OUTPUTS
# ------------------------------------------------------------

excel_path = os.path.join(
    OUTPUT_DIR,
    "IDMC_Repository_Analysis.xlsx"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "migration_run_summary.json"
)

for path in [excel_path, summary_path]:

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    if os.path.getsize(path) == 0:
        raise RuntimeError(f"Empty output file: {path}")


# ------------------------------------------------------------
# 11. FINAL SUMMARY
# ------------------------------------------------------------

success_count = sum(
    1 for item in migration_results
    if item.get("success") is True
)

approval_count = sum(
    1 for item in migration_results
    if item.get("human_approval_verified") is True
)

executed_count = sum(
    1 for item in migration_results
    if item.get("executed") is True
)

print("\n" + "=" * 90)
print("FINAL MIGRATION REPORT")
print("=" * 90)

print("Mappings processed:", len(migration_results))
print("Successful mappings:", success_count)
print("Human approvals verified:", approval_count)
print("Executed mappings:", executed_count)

print("Dry run:", DRY_RUN)
print("Execution approved:", EXECUTION_APPROVED)

print("\nExcel report:", excel_path)
print("JSON summary:", summary_path)

print("\nMIGRATION AND REPORT GENERATION COMPLETED")
